# Ten-column boundary representation

This study asks how an additive boosting model should represent an interaction whose sign switches across a boundary, when the interacting pair is hidden among ten predictors, and whether the data-only specification search finds a representation that predicts as well as CatBoost. Eight methods are compared on the same draws. Each of 50 independent confirmation draws contains 2,000 observations with ten independent Uniform(−1,1) predictors and response $Y=3\operatorname{sign}(x_0x_1)+x_2+\epsilon$, where $\epsilon\sim N(0,0.5^2)$. Each draw is split 70% for fitting and 30% for evaluation (`random_state=0`). Normalized root mean squared error (NRMSE) divides evaluation RMSE by the evaluation response population standard deviation.

The [protocol](all_column_boundary/PROTOCOL.md) fixes all eight methods before measurement. CatBoost with 200 trees is the primary reference. The target is a paired NRMSE excess over CatBoost of at most 1% (excess: 100 × (method NRMSE / CatBoost NRMSE − 1), averaged over draws); a point estimate and a one-sided 95% upper bound are reported separately. Ten development draws and 50 fresh confirmation draws are reported separately. No test result chooses a representation or changes an estimator.

## Predictor representations

`bbs(x)` is a penalized B-spline smooth of one predictor, `bols(x)` a linear effect, and `btree(...)` a regression tree on the named predictors, refitted to the current residual at each boosting update in which it is selected; `selection=sse` makes each split choose the column and cut point that most reduce squared error. The all-column hybrid combines all ten default `bbs` smooths with `btree(x0, ..., x9, maxdepth=6)` for 500 updates at learning rate 0.1. The additive control removes the tree. Two supplied controls restrict the tree to `x0,x1`, depth 3 and `selection=sse`, with either `bols(x2)` or all ten smooths. The supplied tuned target uses the pair plus `bols(x2)`, and chooses the number of updates by five-fold cross-validation at rate 0.1, over a grid that starts at 100 updates and is doubled while the minimum is at its end, up to 500. These supplied formulas use knowledge of the generating mechanism.

Data-only discovery receives all ten predictors, no family or formula, seed 0 and its default limit of 400 boosting updates. It must identify the useful interaction itself. CatBoost uses 200 trees, depth 6 and learning rate 0.05; LightGBM uses 200 trees, 31 leaves, unlimited depth and rate 0.05. Both competitors use seed 0 and one thread. Their settings are fixed rather than tuned to these test draws.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
from docs.book.cases.bench_tables import table

directory = root / "docs/book/benchmarks/all_column_boundary"
summary = json.loads((directory / "confirmation_summary.json").read_text())
facts = json.loads((directory / "confirmation_validation.json").read_text())
labels = {"original_hybrid": "All-column hybrid", "additive": "Additive smooths",
          "supplied_pair": "Supplied pair", "supplied_pair_all_mains": "Pair and all main effects",
          "supplied_tuned": "Supplied tuned pair", "discovery": "Data-only discovery",
          "catboost": "CatBoost 200", "lightgbm": "LightGBM 200"}

In [2]:
headline = pl.DataFrame([{
    "Method": labels[r["method"]], "NRMSE": r["nrmse"], "NRMSE SE": r["nrmse_se"],
    "Mean MSE": r["true_mean_mse"], "Seconds": r["elapsed"], "Failures": r["failures"]}
    for r in summary])
table(headline, title="Every method on the ten-column boundary task",
      subtitle="50 fresh draws; smaller NRMSE, mean MSE and elapsed seconds are better",
      stub="Method", stubhead="Method", better={"NRMSE": "min", "Mean MSE": "min", "Seconds": "min"},
      spanners={"Prediction error": ["NRMSE", "NRMSE SE", "Mean MSE"]},
      notes=["NRMSE SE is the Monte Carlo standard error across independent draws. Mean MSE compares the "
             "prediction with the generating mean in squared response units. Seconds measure the complete "
             "workflow on shared hardware, including tuning for tuned/discovery methods; they are not controlled "
             "implementation speed ratios. Bold marks descriptive minima, not a post hoc method selection.",
             "Failures count incomplete draws; a method with a failure would have no successful-only average. "
             "Source: confirmation_summary.json, which summarize_all_column_boundary.py computes from the 400 "
             "per-draw records written by study_all_column_boundary.py."])

GT(_tbl_data=shape: (8, 6)
┌───────────────────────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ Method                    ┆ NRMSE    ┆ NRMSE SE ┆ Mean MSE ┆ Seconds  ┆ Failures │
│ ---                       ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str                       ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ i64      │
╞═══════════════════════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ All-column hybrid         ┆ 0.329918 ┆ 0.005283 ┆ 0.806926 ┆ 0.365815 ┆ 0        │
│ Additive smooths          ┆ 0.992485 ┆ 0.001325 ┆ 9.200912 ┆ 0.021697 ┆ 0        │
│ Supplied pair             ┆ 0.197671 ┆ 0.002373 ┆ 0.127707 ┆ 0.050574 ┆ 0        │
│ Pair and all main effects ┆ 0.196464 ┆ 0.002274 ┆ 0.12435  ┆ 0.091979 ┆ 0        │
│ Supplied tuned pair       ┆ 0.194045 ┆ 0.002693 ┆ 0.116231 ┆ 0.162648 ┆ 0        │
│ Data-only discovery       ┆ 0.194972 ┆ 0.002532 ┆ 0.120455 ┆ 6.584277 ┆ 0        │
│ CatBoost 200              ┆ 0.216562 ┆ 0.001983 ┆ 0.202496 ┆ 0.461899 ┆ 0        │
│ LightGBM 200              ┆ 0.31192  ┆ 0.003814 ┆ 0.687702 ┆ 0.142214 ┆ 0        │
└───────────────────────────┴──────────┴──────────┴──────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10b551100>, _boxhead=Boxhead([ColInfo(var='Method', type=<ColInfoTypeEnum.stub: 2>, column_label='Method', column_align='left', column_width=None, stub_level=0), ColInfo(var='NRMSE', type=<ColInfoTypeEnum.default: 1>, column_label='NRMSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='NRMSE SE', type=<ColInfoTypeEnum.default: 1>, column_label='NRMSE SE', column_align='right', column_width=None, stub_level=None), ColInfo(var='Mean MSE', type=<ColInfoTypeEnum.default: 1>, column_label='Mean MSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='Seconds', type=<ColInfoTypeEnum.default: 1>, column_label='Seconds', column_align='right', column_width=None, stub_level=None), ColInfo(var='Failures', type=<ColInfoTypeEnum.default: 1>, column_label='Failures', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10cccea50>, _spanners=Spanners([SpannerInfo(spanner_id='Prediction error', spanner_level=0, spanner_label='Prediction error', spanner_units=None, spanner_pattern=None, vars=['NRMSE', 'NRMSE SE', 'Mean MSE'], built=None)]), _heading=Heading(title='Every method on the ten-column boundary task', subtitle='50 fresh draws; smaller NRMSE, mean MSE and elapsed seconds are better', preheader=None), _stubhead='Method', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10b513860>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10ae9dca0>, _source_notes=['NRMSE SE is the Monte Carlo standard error across independent draws. Mean MSE compares the prediction with the generating mean in squared response units. Seconds measure the complete workflow on shared hardware, including tuning for tuned/discovery methods; they are not controlled implementation speed ratios. Bold marks descriptive minima, not a post hoc method selection.', 'Failures count incomplete draws; a method with a failure would have no successful-only average. Source: confirmation_summary.json, which summarize_all_column_boundary.py computes from the 400 per-draw records written by study_all_column_boundary.py.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b2d30b0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10b513bc0>, <great_tables._gt_data.FormatInfo object at 0x10ca6f950>, <great_tables._gt_data.FormatInfo object at 0x10d03f0e0>, <great_tables._gt_data.FormatInfo object at 0x10d03ed20>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10d03f050>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', val

In [3]:
by_method = {r["method"]: r for r in summary}
comparison = pl.DataFrame([{
    "Method": labels[r["method"]], "Excess %": r["catboost_excess"],
    "Paired SE": r["catboost_se"], "Upper 95%": r["catboost_upper95"],
    "Adjusted upper": r["catboost_upper_adjusted95"], "Target": r["upper_target"]}
    for r in summary])
table(comparison, title="Paired NRMSE excess over prespecified CatBoost",
      subtitle="Percentage points; smaller is better; target upper bound ≤ 1%",
      stub="Method", stubhead="Method", better={"Excess %": "min"},
      notes=["Each draw contributes 100 × (method NRMSE / CatBoost NRMSE − 1). Bounds use Student-t over 50 "
             "independent paired draws. Adjusted upper uses the two-comparator Bonferroni critical value; "
             "the separate LightGBM comparison is retained in the summary. Target is the unadjusted upper criterion. "
             "The prespecified target procedures are supplied tuned pair and data-only discovery."])

GT(_tbl_data=shape: (8, 6)
┌───────────────────────────┬────────────┬───────────┬────────────┬────────────────┬────────┐
│ Method                    ┆ Excess %   ┆ Paired SE ┆ Upper 95%  ┆ Adjusted upper ┆ Target │
│ ---                       ┆ ---        ┆ ---       ┆ ---        ┆ ---            ┆ ---    │
│ str                       ┆ f64        ┆ f64       ┆ f64        ┆ f64            ┆ bool   │
╞═══════════════════════════╪════════════╪═══════════╪════════════╪════════════════╪════════╡
│ All-column hybrid         ┆ 52.783763  ┆ 2.657846  ┆ 57.239777  ┆ 58.124904      ┆ false  │
│ Additive smooths          ┆ 360.130251 ┆ 4.156979  ┆ 367.099639 ┆ 368.484014     ┆ false  │
│ Supplied pair             ┆ -8.596057  ┆ 0.994283  ┆ -6.92909   ┆ -6.59797       ┆ true   │
│ Pair and all main effects ┆ -9.098138  ┆ 1.057012  ┆ -7.326004  ┆ -6.973993      ┆ true   │
│ Supplied tuned pair       ┆ -10.283144 ┆ 1.149462  ┆ -8.356012  ┆ -7.973213      ┆ true   │
│ Data-only discovery       ┆ -9.811061  ┆ 1.135828  ┆ -7.906788  ┆ -7.52853       ┆ true   │
│ CatBoost 200              ┆ 0.0        ┆ 0.0       ┆ 0.0        ┆ 0.0            ┆ true   │
│ LightGBM 200              ┆ 44.611014  ┆ 2.183806  ┆ 48.272276  ┆ 48.999536      ┆ false  │
└───────────────────────────┴────────────┴───────────┴────────────┴────────────────┴────────┘, _body=<great_tables._gt_data.Body object at 0x10ae91df0>, _boxhead=Boxhead([ColInfo(var='Method', type=<ColInfoTypeEnum.stub: 2>, column_label='Method', column_align='left', column_width=None, stub_level=0), ColInfo(var='Excess %', type=<ColInfoTypeEnum.default: 1>, column_label='Excess %', column_align='right', column_width=None, stub_level=None), ColInfo(var='Paired SE', type=<ColInfoTypeEnum.default: 1>, column_label='Paired SE', column_align='right', column_width=None, stub_level=None), ColInfo(var='Upper 95%', type=<ColInfoTypeEnum.default: 1>, column_label='Upper 95%', column_align='right', column_width=None, stub_level=None), ColInfo(var='Adjusted upper', type=<ColInfoTypeEnum.default: 1>, column_label='Adjusted upper', column_align='right', column_width=None, stub_level=None), ColInfo(var='Target', type=<ColInfoTypeEnum.default: 1>, column_label='Target', column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b513a40>, _spanners=Spanners([]), _heading=Heading(title='Paired NRMSE excess over prespecified CatBoost', subtitle='Percentage points; smaller is better; target upper bound ≤ 1%', preheader=None), _stubhead='Method', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d03f680>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d03f5f0>, _source_notes=['Each draw contributes 100 × (method NRMSE / CatBoost NRMSE − 1). Bounds use Student-t over 50 independent paired draws. Adjusted upper uses the two-comparator Bonferroni critical value; the separate LightGBM comparison is retained in the summary. Target is the unadjusted upper criterion. The prespecified target procedures are supplied tuned pair and data-only discovery.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10d079c70>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10d03f710>, <great_tables._gt_data.FormatInfo object at 0x10d079f70>, <great_tables._gt_data.FormatInfo object at 0x10d07a000>, <great_tables._gt_data.FormatInfo object at 0x10d079160>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10d079b20>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, ca

The supplied tuned pair has a mean paired NRMSE excess of {eval}`f"{by_method['supplied_tuned']['catboost_excess']:.2f}%"`, with upper bound {eval}`f"{by_method['supplied_tuned']['catboost_upper95']:.2f}%"`. Data-only discovery has excess {eval}`f"{by_method['discovery']['catboost_excess']:.2f}%"`, with upper bound {eval}`f"{by_method['discovery']['catboost_upper95']:.2f}%"`. Both prespecified workflows meet the target on this task. The all-column hybrid is worse, at {eval}`f"{by_method['original_hybrid']['catboost_excess']:.2f}%"` excess.

A `btree` term already adds a new tree at each update in which it is selected, so the boosted model is a tree ensemble; the gap of the all-column hybrid does not come from fitting a single tree. This experiment supports the pair representation and the data-only selection workflow for this generating process; other generating processes may need other tree depths or representations.

## Public fitting workflow

The example repeats one frozen confirmation input to illustrate fitting, stopping selection and prediction. It adds no independent evidence to the comparison. The five inner folds choose the number of updates using fitting rows only; the evaluation response is used after prediction.


In [4]:
import mboostrs as mb
from benches.study_all_column_boundary import draw

frame, true_mean, fitting_rows, testing_rows = draw(6088100)
fitting = frame[fitting_rows]
evaluation = frame[testing_rows]

selected = mb.tune_mstop(
    "y ~ btree(x0, x1, maxdepth=3, selection=sse) + bols(x2)", fitting,
    family=mb.Gaussian(), nu=0.1, folds=5, start=100, max_mstop=500, seed=0)
prediction = selected.model.predict(evaluation.drop("y"), type="response")
example_mse = float(np.mean((prediction - evaluation["y"].to_numpy()) ** 2))
example = pl.DataFrame({"Observed": evaluation["y"].head(5), "Predicted": prediction[:5]})
(GT(example).tab_header(title="Supplied tuned predictions", subtitle="Response units; first five evaluation observations")
 .fmt_number(columns=["Observed", "Predicted"], decimals=3)
 .tab_source_note(f"Selected updates: {selected.mstop}; evaluation MSE: {example_mse:.4f}. "
                  "Demonstration on the first confirmation draw."))

GT(_tbl_data=shape: (5, 2)
┌───────────┬───────────┐
│ Observed  ┆ Predicted │
│ ---       ┆ ---       │
│ f64       ┆ f64       │
╞═══════════╪═══════════╡
│ -1.680408 ┆ -2.055892 │
│ -3.071334 ┆ -3.214441 │
│ 2.881683  ┆ 3.444997  │
│ 2.153337  ┆ 1.919386  │
│ 3.289859  ┆ 2.928678  │
└───────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x120c2dcd0>, _boxhead=Boxhead([ColInfo(var='Observed', type=<ColInfoTypeEnum.default: 1>, column_label='Observed', column_align='right', column_width=None, stub_level=None), ColInfo(var='Predicted', type=<ColInfoTypeEnum.default: 1>, column_label='Predicted', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x125faab10>, _spanners=Spanners([]), _heading=Heading(title='Supplied tuned predictions', subtitle='Response units; first five evaluation observations', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x125fceae0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x122a48d10>, _source_notes=['Selected updates: 155; evaluation MSE: 0.3645. Demonstration on the first confirmation draw.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x126046000>, _formats=[<great_tables._gt_data.FormatInfo object at 0x12559e060>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'

The corresponding data-only call is `mb.discover(fitting, "y", seed=0)`. A returned distributional model reports a dictionary of predictions, from which the mean is `prediction["mu"]`; a standard model returns the mean array directly. The following table shows the formula and family the search selected on the confirmation input used in the preceding example.

In [5]:
# The summarizer copies the data-only choices on the first confirmation draw into the discovery row.
choices = by_method["discovery"]["first_draw_selection"]
choice_frame = pl.DataFrame({"Field": ["Seed", "Family", "Mean formula", "Spread formula", "Stopping budget"],
    "Value": [str(choices["seed"]), str(choices["family"]), str(choices["formula"]),
              str(choices["spread_formula"]), str(choices["mstop"])]})
(GT(choice_frame).tab_header(title="First frozen data-only selection",
                           subtitle="The search sees all ten observed predictors")
 .tab_source_note("Source: confirmation_summary.json. The per-draw record in confirmation_a.jsonl.xz also holds the full "
                  "search log, provenance, predictions, warnings and row identities."))

GT(_tbl_data=shape: (5, 2)
┌─────────────────┬─────────────────────────────────┐
│ Field           ┆ Value                           │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ Seed            ┆ 6088100                         │
│ Family          ┆ Gaussian                        │
│ Mean formula    ┆ bbs(x2, df=4.0) + bbs(x9, df=4… │
│ Spread formula  ┆ None                            │
│ Stopping budget ┆ 201                             │
└─────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x1254105c0>, _boxhead=Boxhead([ColInfo(var='Field', type=<ColInfoTypeEnum.default: 1>, column_label='Field', column_align='left', column_width=None, stub_level=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x1253496d0>, _spanners=Spanners([]), _heading=Heading(title='First frozen data-only selection', subtitle='The search sees all ten observed predictors', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d03eab0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d03ea50>, _source_notes=['Source: confirmation_summary.json. The per-draw record in confirmation_a.jsonl.xz also holds the full search log, provenance, predictions, warnings and row identities.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10d03f170>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='tab

## Evidence and validation

Development used seeds 6087100–6087109 and confirmation seeds 6088100–6088149; the seed sets are disjoint. The two confirmation archives contain disjoint batches of 25 draws. Every method within a draw shares the fitting/evaluation rows and the same generated data. The validator checks compressed and uncompressed SHA-256 digests, exactly eight records per draw, 1,400/600 disjoint row identities, and recomputes MSE, latent-mean MSE (the MSE of the predicted mean against the noiseless latent mean) and NRMSE from stored predictions to absolute tolerance $10^{-12}$.

All failures, warnings, model choices, elapsed costs and source/native identities are retained. The 80 development records and all 400 confirmation records completed.

The study does not test the noise-scale (standard deviation) model that the search may select: accurate mean prediction does not show that a selected spread formula is correct.

In [6]:
development = json.loads((directory / "development_summary.json").read_text())
phases = pl.DataFrame([{"Phase": phase, "Method": labels[r["method"]], "NRMSE": r["nrmse"],
                       "SE": r["nrmse_se"], "Failures": r["failures"]}
    for phase, records in (("Development (10)", development), ("Confirmation (50)", summary)) for r in records])
table(phases, title="Development and confirmation reported separately",
      subtitle="NRMSE and Monte Carlo SE; lower NRMSE is better", stub="Method", stubhead="Method", group="Phase",
      better={"NRMSE": "min"}, notes=["Source: development_summary.json and confirmation_summary.json. "
      "No confirmation result changes the predefined methods or budgets; failures are not excluded from aggregates."])

GT(_tbl_data=shape: (16, 5)
┌───────────────────┬───────────────────────────┬──────────┬──────────┬──────────┐
│ Phase             ┆ Method                    ┆ NRMSE    ┆ SE       ┆ Failures │
│ ---               ┆ ---                       ┆ ---      ┆ ---      ┆ ---      │
│ str               ┆ str                       ┆ f64      ┆ f64      ┆ i64      │
╞═══════════════════╪═══════════════════════════╪══════════╪══════════╪══════════╡
│ Development (10)  ┆ All-column hybrid         ┆ 0.322971 ┆ 0.013953 ┆ 0        │
│ Development (10)  ┆ Additive smooths          ┆ 0.991579 ┆ 0.002346 ┆ 0        │
│ Development (10)  ┆ Supplied pair             ┆ 0.21005  ┆ 0.006603 ┆ 0        │
│ Development (10)  ┆ Pair and all main effects ┆ 0.204533 ┆ 0.005403 ┆ 0        │
│ Development (10)  ┆ Supplied tuned pair       ┆ 0.20605  ┆ 0.006894 ┆ 0        │
│ …                 ┆ …                         ┆ …        ┆ …        ┆ …        │
│ Confirmation (50) ┆ Pair and all main effects ┆ 0.196464 ┆ 0.002274 ┆ 0        │
│ Confirmation (50) ┆ Supplied tuned pair       ┆ 0.194045 ┆ 0.002693 ┆ 0        │
│ Confirmation (50) ┆ Data-only discovery       ┆ 0.194972 ┆ 0.002532 ┆ 0        │
│ Confirmation (50) ┆ CatBoost 200              ┆ 0.216562 ┆ 0.001983 ┆ 0        │
│ Confirmation (50) ┆ LightGBM 200              ┆ 0.31192  ┆ 0.003814 ┆ 0        │
└───────────────────┴───────────────────────────┴──────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x12659e180>, _boxhead=Boxhead([ColInfo(var='Phase', type=<ColInfoTypeEnum.row_group: 3>, column_label='Phase', column_align='left', column_width=None, stub_level=None), ColInfo(var='Method', type=<ColInfoTypeEnum.stub: 2>, column_label='Method', column_align='left', column_width=None, stub_level=0), ColInfo(var='NRMSE', type=<ColInfoTypeEnum.default: 1>, column_label='NRMSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='SE', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None, stub_level=None), ColInfo(var='Failures', type=<ColInfoTypeEnum.default: 1>, column_label='Failures', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x125bb8fe0>, _spanners=Spanners([]), _heading=Heading(title='Development and confirmation reported separately', subtitle='NRMSE and Monte Carlo SE; lower NRMSE is better', preheader=None), _stubhead='Method', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12659e690>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12659e240>, _source_notes=['Source: development_summary.json and confirmation_summary.json. No confirmation result changes the predefined methods or budgets; failures are not excluded from aggregates.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12659d580>, _formats=[<great_tables._gt_data.FormatInfo object at 0x125faaba0>, <great_tables._gt_data.FormatInfo object at 0x12419d2b0>, <great_tables._gt_data.FormatInfo object at 0x12659e630>, <great_tables._gt_data.FormatInfo object at 0x12659ea20>, <great_tables._gt_data.FormatInfo object at 0x12659e8d0>, <great_tables._gt_data.FormatInfo object at 0x12659ea50>, <great_tables._gt_data.FormatInfo object at 0x12659ec30>, <great_tables._gt_data.FormatInfo object at 0x12659e930>, <great_tables._gt_data.FormatInfo object at 0x12659e900>, <great_tables._gt_data.FormatInfo object at 0x12659ee10>, <great_tables._gt_data.FormatInfo object at 0x12659ec90>, <great_tables._gt_data.FormatInfo object at 0x12659ef00>, <great_tables._gt_data.FormatInfo object at 0x12659efc0>, <great_tables._gt_data.FormatInfo object at 0x12659ef90>, <great_tables._gt_data.FormatInfo object at 0x12659f260>, <great_tables._gt_data.FormatInfo object at 0x12659f2c0>, <great_tables._gt_data.FormatInfo object at 0x12659f4a0>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x12659f410>], _col_merge=[], _transforms=[], _options=Op

## Reproduction

Use the repository-local Python environment and a release extension matching the source revision recorded in each metadata file. The supplied runner refuses to overwrite evidence. Choose a new output directory or preserve the existing files before a reproduction. From the repository root:

```bash
PYTHONPATH=python:. .venv/bin/python benches/study_all_column_boundary.py \
  --phase development --output /tmp/boundary-development.jsonl.xz
PYTHONPATH=python:. .venv/bin/python benches/study_all_column_boundary.py \
  --phase confirmation --output /tmp/boundary-confirmation.jsonl.xz
PYTHONPATH=python:. .venv/bin/python benches/summarize_all_column_boundary.py --phase confirmation
```

The final command validates the per-draw archives, which must first be downloaded as described below. Their metadata records the runner hash, loaded Python source hashes, native build identity, package versions, thread settings and protocol hash. Reproducing on newer code is a new run and does not replace the frozen result. The downloadable notebook bundle includes the protocol, metadata, summary and validation files listed below. The per-draw archives are assets of the `benchmark-raw-archives-1` GitHub release; `docs/book/benchmarks/raw_archives.json` lists each file with its size, SHA-256 and the command that regenerates it. They are not bundled.

In [7]:
# Per-draw archives are release assets listed in raw_archives.json; the bundle holds the files this page reads or cites.
artifacts = sorted(p.name for p in directory.iterdir()
                   if p.is_file() and not p.name.endswith((".jsonl.xz", ".jsonl.gz")))
(GT(pl.DataFrame({"Artifact": artifacts}))
 .tab_header(title="Bundled evidence", subtitle="Protocol, run metadata, summaries and validation")
 .tab_source_note("Each metadata file records the SHA-256 digests of one per-draw archive; the summarizer checks "
                  "them before recomputing every score from stored predictions."))

Bundled evidence
"Protocol, run metadata, summaries and validation"
Artifact
PROTOCOL.md
VALIDATION.md
confirmation_a.jsonl.metadata.json
confirmation_b.jsonl.metadata.json
confirmation_summary.json
confirmation_validation.json
development.jsonl.metadata.json
development_summary.json
